# Küçük sayılarla RSA
p=3, q=11, e=3, d=7. Türk alfabesi indeksleri kullanılır.


In [ ]:
from math import gcd

ALPHABET = "abcçdefgğhıijklmnoöprsştuüvyz"

def encode_text(text):
    values = []
    for character in text.lower():
        if character not in ALPHABET:
            raise ValueError(f"Bu örnekte desteklenmeyen karakter: {character!r}")
        values.append(ALPHABET.index(character))
    return values

def decode_values(values):
    if any(not isinstance(value, int) or not 0 <= value < len(ALPHABET) for value in values):
        raise ValueError("Çözülen değer Türk alfabesi aralığında değil.")
    return "".join(ALPHABET[value] for value in values)

def make_keys(p=3, q=11, e=3):
    n = p * q
    phi = (p - 1) * (q - 1)
    if gcd(e, phi) != 1:
        raise ValueError("e ile φ(n) aralarında asal olmalı.")
    d = pow(e, -1, phi)
    return dict(p=p, q=q, n=n, phi=phi, e=e, d=d)

def encrypt(text, keys):
    return [pow(value, keys["e"], keys["n"]) for value in encode_text(text)]

def decrypt(ciphertext, keys):
    values = [pow(value, keys["d"], keys["n"]) for value in ciphertext]
    return decode_values(values)

def power_steps(base, exponent, modulus):
    result = 1
    trace = []
    for bit in bin(exponent)[2:]:
        before = result
        squared = before * before
        square_remainder = squared % modulus
        product = square_remainder * base if bit == "1" else None
        result = product % modulus if product is not None else square_remainder
        trace.append(dict(bit=bit, before=before, squared=squared,
                          square_remainder=square_remainder,
                          product=product, after=result))
    return result, trace


## 1. Doğrudan şifrele ve çöz
Küçük anahtarlarla sonucu görelim.


In [ ]:
text = "merhaba"
keys = make_keys(p=3, q=11, e=3)
ciphertext = encrypt(text, keys)
recovered = decrypt(ciphertext, keys)
print("Açık metin   :", text)
print("Açık anahtar :", (keys["n"], keys["e"]))
print("Özel anahtar:", (keys["n"], keys["d"]))
print("Şifreli      :", ciphertext)
print("Çözülmüş     :", recovered)
assert recovered == text

Açık metin   : merhaba
Açık anahtar : (33, 3)
Özel anahtar: (33, 7)
Şifreli      : [9, 26, 14, 3, 0, 1, 0]
Çözülmüş     : merhaba


## 2. Harfleri küçük sayılara çevir
n=33 olduğu için UTF-8 baytları yerine Türk alfabesindeki 0 tabanlı indeksleri kullanıyoruz.


In [ ]:
values = encode_text(text)
for character, value in zip(text, values):
    print(f"{character} → {value}")
print("Sayı dizisi:", values)
assert all(0 <= value < keys["n"] for value in values)

m → 15
e → 5
r → 20
h → 9
a → 0
b → 1
a → 0
Sayı dizisi: [15, 5, 20, 9, 0, 1, 0]


## 3. Anahtarları üret
p=3 ve q=11 ile bütün değerler elle takip edilebilir.


In [ ]:
p, q, e = 3, 11, 3
n = p * q
phi = (p - 1) * (q - 1)
d = pow(e, -1, phi)
print(f"n = {p} × {q} = {n}")
print(f"φ(n) = ({p}−1) × ({q}−1) = {phi}")
print(f"e = {e}")
print(f"d = {d}; çünkü {e} × {d} = {e*d} = 1 + {phi}")
print("Açık anahtar :", (n, e))
print("Özel anahtar:", (n, d))

n = 3 × 11 = 33
φ(n) = (3−1) × (11−1) = 20
e = 3
d = 7; çünkü 3 × 7 = 21 = 1 + 20
Açık anahtar : (33, 3)
Özel anahtar: (33, 7)


## 4. İlk harfi şifrele
m harfinin alfabe indeksi 15. Şifreleme c = mᵉ mod n.


In [ ]:
message = values[0]
encrypted = pow(message, e, n)
print(f"c = {message}^{e} mod {n}")
print(f"  = {message**e} mod {n}")
print(f"  = {encrypted}")
assert encrypted == ciphertext[0] == 9

c = 15^3 mod 33
  = 3375 mod 33
  = 9


## 5. Bütün harfleri şifrele
Her sayı aynı açık anahtarla şifrelenir.


In [ ]:
for character, message, encrypted in zip(text, values, ciphertext):
    print(f"{character}: {message}^{e} mod {n} = {encrypted}")
print("Şifreli dizi:", ciphertext)

m: 15^3 mod 33 = 9
e: 5^3 mod 33 = 26
r: 20^3 mod 33 = 14
h: 9^3 mod 33 = 3
a: 0^3 mod 33 = 0
b: 1^3 mod 33 = 1
a: 0^3 mod 33 = 0
Şifreli dizi: [9, 26, 14, 3, 0, 1, 0]


## 6. Özel anahtarla çöz
Çözme m = cᵈ mod n. Burada d=7.


In [ ]:
decoded = []
for encrypted in ciphertext:
    message = pow(encrypted, d, n)
    decoded.append(message)
    print(f"{encrypted}^{d} mod {n} = {message}")
print("Çözülen sayılar:", decoded)
print("Metin:", decode_values(decoded))
assert decoded == values

9^7 mod 33 = 15
26^7 mod 33 = 5
14^7 mod 33 = 20
3^7 mod 33 = 9
0^7 mod 33 = 0
1^7 mod 33 = 1
0^7 mod 33 = 0
Çözülen sayılar: [15, 5, 20, 9, 0, 1, 0]
Metin: merhaba
